# Coding Session #6

---

## Today's session

Today's session is structured to teach more complex tasks. We will explore:

- Real-world RQ as a campaigner
- More advanced functions
- Permutation test

### 0.1 Environment preparation

We begin by loading the libraries we’ll need. In Python, libraries are like toolkits: they extend the language with specialized functions.

- **pandas (pd)** is our main tool for working with tabular data. It introduces the DataFrame, which lets us manipulate datasets in a way that feels natural if you’ve used Excel or R.
- **NumPy (np)** provides the numerical backbone. It gives us arrays and fast mathematical functions, which pandas actually uses under the hood.
- **Seaborn (sns)** builds on top of Matplotlib to create a vast range of plots and visualization.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns


### 0.2 Reading Dataset


The **Database on Ideology, Money in Politics, and Elections (DIME), Version 4.0** provides a comprehensive record of campaign finance activity in the United States between **1979 and 2024**. It contains more than **850 million itemized contributions** from individuals and organizations to candidates, parties, and political committees at the federal, state, and local levels (Bonica, 2024).

DIME is organized into two primary components:

* **Candidate/Recipient Database** – includes information on candidates and committees, such as names, party affiliation, state, office sought, district, gender, incumbency status, fundraising totals, election outcomes, and multiple measures of ideology.
* **Contributor Database** – contains records on donors, both individuals and organizations, with demographic details, contribution histories, and ideological scores.

For this coding session, we use a **subset of the candidate/recipient database** covering the **2020 and 2024 federal elections** with some small modifications.

You can access the complete dataset here: [DIME 4.0 – Stanford University Libraries](http://data.stanford.edu/dime).

**Reference:**
Bonica, Adam. 2024. *Database on Ideology, Money in Politics, and Elections: Public version 4.0* [Computer file]. Stanford, CA: Stanford University Libraries. Available at: [http://data.stanford.edu/dime](http://data.stanford.edu/dime).

#### Codebook


| Variable                | Description |
|--------------------------|-------------|
| **lname**               | Last name of the candidate/recipient. |
| **fname**               | First name of the candidate/recipient. |
| **party**               | Party of candidate/recipient (100 = Democrat, 200 = Republican, 328 = Independent). |
| **state**               | Two-letter state abbreviation. |
| **seat**                | Office sought (e.g., `federal:house`, `federal:senate`, `federal:president`, `state:governor`, etc.). |
| **district**            | District code: two-letter state code followed by congressional district number. For Senate candidates, “S” plus the year of the seat’s election. |
| **ico.status**          | Incumbency status (`I` = Incumbent, `C` = Challenger, `O` = Open Seat Candidate, blank = not up for election). |
| **cand.gender**         | Candidate gender coding (based on Census first-name ratios and gendered titles). |
| **recipient.cfscore.dyn** | Period-specific estimates of ideology. Candidate/recipient scores are re-estimated each cycle while holding contributor scores constant. |
| **contributor.cfscore** | Estimated ideology of the candidate/recipient based on their **personal donations** to other candidates/recipients. |
| **num.givers.total**    | Number of distinct donors that gave to the candidate/recipient over their career. |
| **total.receipts**      | Sum total of contributions raised during an election cycle. |
| **total.disbursements**      | Sum total of disbursements spent during an election cycle. |
| **prim.vote.pct**       | FEC-reported vote share (%) in the primary election (federal congressional candidates only). |
| **pwinner**             | Primary election outcome (`W` = won; `L` = lost). Federal congressional candidates only. |
| **gen.vote.pct**        | FEC-reported vote share (%) in the general election. |
| **gwinner**             | General election outcome (`W` = won; `L` = lost). Federal candidates coded from FEC, state candidates from NIMSP. |



In [ ]:
#Load "dime_candidates_2020_2024.csv" from remote repository
df_bonica = pd.read_csv("https://raw.githubusercontent.com/albertostefanelli/DSPC_coding_sessions/refs/heads/master/codingsession06/data/dime_candidates_2020_2024_clean.csv")

# Quick check
df_bonica.head()
df_bonica.dtypes

## 1. Are ideologically moderate candidates better fundraisers than extremists?

Campaign managers constantly face a strategic trade-off:
- Moderates can appeal to swing voters and big donors but might alienate the activist base.
- Extremists can mobilise small donors and grassroots fundraising but struggle with general-election viability.

Understanding which group actually raises more money---and from what kinds of donors---directly informs candidate recruitment, messaging, and fundraising strategy.

Descriptive differences in mean receipts) helps answer such
- Should a party encourage moderate challengers in swing districts?
- Do extremist candidates actually bring in more small-donor money?


We’ll need:
- `cand.cfscore` – ideological score (negative = liberal, positive = conservative)
- `total.receipts` – total amount of money raised
- A threshold that splits candidates into:
 - Moderates (closer to 0)
 - Extremists (farther from 0)



What we are doing:

- We’re comparing the average fundraising of two naturally occurring groups: moderates (1) and extremists (0).
- Technically: unadjusted difference in means, or descriptive gap.
- It’s not an experiment—candidates choose their ideological positions, and ideology may be correlated with many other factors

### Let's take a quick look at the data

We start by exploring the distribution of candidates’ ideology scores to get a sense of how this variable is shaped and what analytical steps might make sense next. Remeber: the CFscore measures how liberal or conservative each candidate is, based on the ideological leanings of their donors and networks.

To analyze fundraising patterns, we often need to categorize candidates into groups, i.e.,  “moderate” vs. “extreme.”

There is no fixed rule for where moderation ends and extremity begins, so the threshold  should be guided by both:

- the distribution of the data (where natural clusters appear), and
- the substantive meaning in political terms (what would typically count as ideologically moderate or extreme within the campaign context).
- prior resarch/campaign decision


In [ ]:
df_bonica_plot = df_bonica[['recipient.cfscore.dyn']].dropna().copy()
df_bonica_plot.rename(columns={'recipient.cfscore.dyn': 'ideology'}, inplace=True)

In [ ]:
hist_ideo = sns.histplot(df_bonica_plot['ideology'],
             bins=40,
             kde=True, color='skyblue')

# --- 3. Add labels and title ---
hist_ideo.set(
    title="Distribution of Candidate Ideology (CFscore)",
    xlabel="Ideological Score (Negative = Liberal, Positive = Conservative)",
    ylabel="Number of Candidates"
)

## 2. Functions (1)

We need to program two simple helper functions that will make our analysis more modular and reusable.

- A function that sets a **threshold for moderates**. This function will take the ideological score (CFscore) and classify candidates as either moderate or extreme based on a chosen threshold around the center (for example, ±0.5).
Candidates with scores between –0.5 and +0.5 will be labeled as moderate (1),
while those outside that range will be labeled as extreme (0).
- A function that computes the mean difference between **moderates and extremes**, we can compare their average fundraising performance. This will be: $$\text{Mean Difference} = \overline{Y}_{\text{moderate}=1} - \overline{Y}_{\text{moderate}=0}$$


In [ ]:
def make_moderate_indicator(df, ideology_col, tau=0.5, colname="moderate"):
    """Add a binary column: 1 if |ideology| <= tau, else 0."""
    df[colname] = (df[ideology_col].abs() <= tau).astype(int)
    return df

In [ ]:
# let's now create this new dummy variable
df_bonica = make_moderate_indicator(df_bonica, "recipient.cfscore.dyn", tau=0.5)

In [ ]:
df_bonica.head()

In [ ]:
def mean_diff(df, group_col, outcome_col):
    """
    Compute the difference in mean outcomes between two groups.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataset containing both the grouping and outcome variables.
    group_col : str
        The name of the binary treatment column (1 = focal group, 0 = comparison group).
    outcome_col : str
        The name of the outcome variable (numeric).

    Returns
    -------
    float
        The difference in mean outcome between the two groups.
        (mean_outcome_focal - mean_outcome_comparison)
    """

    # 1. Keep only the relevant columns and remove missing data
    data = df[[group_col, outcome_col]].dropna()

    # 2. Separate the groups
    focal_group = data.loc[data[group_col] == 1, outcome_col]
    comparison_group = data.loc[data[group_col] == 0, outcome_col]

    # 3. Compute mean outcomes
    mean_focal = focal_group.mean()
    mean_comparison = comparison_group.mean()

    # 4. Compute the difference in means
    diff = mean_focal - mean_comparison

    return diff


Let's now calculate the mean difference



In [ ]:
diff = mean_diff(df_bonica, "moderate", "total.receipts")
diff

- The negative sign means that the group coded as 1 (moderates) raised less money on average than the group coded as 0 (extremists).
- On average, moderate candidates raised about $376,000 less than extremist candidates in this dataset.
- Candidates positioned closer to the ideological center tended to attract less financial support than those at the ideological edges.
- If you’re running a moderate candidate you should be aware of this

## 3. Is this difference large enough that we can be confident it’s not just random noise? (permutation test)

So far, we calculated a difference in means and we know how much moderates and extremists differ in their average fundraising — but not whether that difference could have occurred just by chance.

So far, we calculated a difference in means and we know how much moderates and extremists differ in their average fundraising — but not whether that difference could have occurred just by chance.

**Why do we need a permutation test?**

A permutation test helps us ask:

> “If there were no real relationship between ideology and fundraising — if being moderate or extreme didn’t matter — how big of a difference in means would we expect just by random chance?”

In other words, we simulate a world where moderate vs. extreme don’t actually affect the outcome, and we see whether our observed difference looks unusual in that world.

### How to do this conceptually

1. Start with your data: Each candidate has an ideology label (`moderate`) and an outcome (`total.receipts`).
2. Compute the observed difference
3. Shuffle the labels: Randomly reassign the moderate labels among candidates.
  - This breaks any real relationship between ideology and fundraising.
4. Recalculate the difference in means for this shuffled data.
5. Repeat this many times (e.g., 5,000 times).
  - This gives us a distribution of “differences you’d get by chance.”
6. Compare the observed difference to this null distribution:
  - If your observed difference is far out in the tails, it’s unlikely to be random.
  - If it’s near the center, it could easily occur by chance.

Next week we will see how to compute a p-value using this very approach

In [ ]:
# Let's build a permutation test function

def permutation_test(df, group_col, outcome_col, n_permutations=1000):
    """
    Simple permutation test for the difference in means.

    Steps:
    1. Compute the observed difference in means using mean_diff().
    2. Shuffle the focal labels many times and recompute the difference.
    3. Compare the observed difference to the shuffled differences to get a p-value.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataset containing both the grouping (group_col) and outcome (outcome_col).
    group_col : str
        The column with a binary grouping variable (e.g., 1 = moderates, 0 = extremists).
    outcome_col : str
        The numeric outcome variable (e.g., total.receipts).
    n_permutations : int
        Number of random shuffles to perform (default = 1000).
    """

    # --- Step 1. Compute the observed difference in means ---
    obs_diff = mean_diff(df, group_col, outcome_col)

    # --- Step 2. Prepare an array to store results ---
    perm_diffs = np.empty(n_permutations)

    # --- Step 3. Run the permutation test loop ---
    for i in range(n_permutations):
        # Randomly shuffle the labels (moderate vs. extreme)
        shuffled = np.random.permutation(df[group_col])

        # Create a temporary copy with shuffled labels
        df_shuffled = df.copy()
        df_shuffled[group_col] = shuffled

        # Compute the difference in means for this shuffled data
        perm_diffs[i] = mean_diff(df_shuffled, group_col, outcome_col)


    return obs_diff, perm_diffs



In [ ]:
obs_diff, perm_diffs = permutation_test(df_bonica, "moderate", "total.receipts", n_permutations=1000)

After running the permutation test, we would like to see the distribtion of our premuted draws.

- This shows the null world — what kinds of differences in means we’d expect to see if ideology didn’t matter at all.
- Each bar comes from one random shuffle of the data, where we break any real connection between being moderate or extreme and the amount of money raised.
- We should add a red line marks the actual observed difference in the real data — how much moderates and extremists differed in their real fundraising averages.

In [ ]:
hist_perm = sns.histplot(
            perm_diffs,
            bins=40,
            kde=True,
            color='skyblue'
        )

hist_perm.axvline(obs_diff, color='red', linestyle='--', linewidth=2, label='Observed difference')

# Add labels and title
hist_perm.set(
    title="Distribution of Permuted Differences",
    xlabel=r"Difference in means ($\bar{Y}_{1}-\bar{Y}_{0}$)",
    ylabel="Number of permutation draws"
)

**Looking at this plot, would you say that ideology has a meaningful relationship with fundraising?**

# Congratulations!

You are done with the coding session. Questions or suggestions? Email Alberto at alberto.stefanelli@yale.edu

In [ ]:
# Install requirements
!apt-get -qq update
!apt-get install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic

from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Ask for the notebook name
notebook_name = input(
    "Enter your notebook’s exact file name,\n"
    "exactly as shown in the top-left corner of the Colab page (next to the two yellow circle icons): "
)

# Build paths
input_path = f"/content/drive/MyDrive/Colab Notebooks/{notebook_name}"
output_path = input_path.replace(".ipynb", ".pdf")

# Convert to PDF
!jupyter nbconvert --to pdf "{input_path}"

# Download the PDF
files.download(output_path)